In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torchvision import datasets, models
from tqdm import tqdm

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')

Using device: cuda


In [ ]:
# Load and preprocess the dataset from /data/train and /data/test for val data do a 70/30 split of train data
import os
from pathlib import Path

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

def find_existing_path(candidates):
    for p in candidates:
        if Path(p).exists():
            return str(Path(p))
    return None

# Prefer dataset folders inside this notebook folder, but also try common paths
train_root = find_existing_path([
    './data/train',
    'data/train',
    'portfolio/trash/data/train',
    os.path.join(os.getcwd(), 'portfolio', 'trash', 'data', 'train'),
])
test_root = find_existing_path([
    './data/test',
    'data/test',
    'portfolio/trash/data/test',
    os.path.join(os.getcwd(), 'portfolio', 'trash', 'data', 'test'),
])
print(f'Found train_root: {train_root}, test_root: {test_root}')

if train_root and test_root:
    train_dataset = datasets.ImageFolder(root=train_root, transform=transform)
    test_dataset = datasets.ImageFolder(root=test_root, transform=transform)
    num_classes = len(train_dataset.classes)
else:
    # Fallback to FakeData so the notebook runs even if local data folders are missing.
    # Adjust num_classes or replace with real paths if available.
    print("Warning: '/data/train' or '/data/test' not found. Using FakeData fallback for debugging.")
    num_classes = 2
    train_dataset = datasets.FakeData(size=1000, image_size=(3, 224, 224), num_classes=num_classes, transform=transform)
    test_dataset = datasets.FakeData(size=200, image_size=(3, 224, 224), num_classes=num_classes, transform=transform)

train_size = int(0.7 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [train_size, val_size])
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

#define a resnet18 model for image classification
model = models.resnet18(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes)
model = model.to(device)
# Define loss function and optimizer with early stopping
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
best_val_loss = float('inf')
patience = 5
counter = 0
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in tqdm(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
    val_loss /= len(val_loader.dataset)
    print(f'Epoch {epoch+1}/{num_epochs}, Training Loss: {epoch_loss:.4f}, Validation Loss: {val_loss:.4f}')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        counter += 1
        if counter >= patience:
            print('Early stopping')
            break
# Load the best model and evaluate on test data
model.load_state_dict(torch.load('best_model.pth'))
model.eval()
test_loss = 0.0
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        test_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
test_loss /= len(test_loader.dataset)
accuracy = 100 * correct / total
print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {accuracy:.2f}%')



FileNotFoundError: Couldn't find any class folder in data\train.